# Low-Entropy LLM Watermarking

**Baseline**: Soft red-list watermark from Kirchenbauer et al. (ICML 2023)  
Paper: https://arxiv.org/abs/2301.10226  
Original code: https://github.com/jwkirchenbauer/lm-watermarking

This notebook is the interactive entry-point. The core logic lives in:
- `watermark.py` — `WatermarkLogitsProcessor` + `WatermarkDetector`
- `generate.py` — bulk generation on C4 RealNewsLike
- `evaluate.py` — z-scores, PPL, ROC/AUC

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from watermark import WatermarkLogitsProcessor, WatermarkDetector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
GAMMA = 0.25
DELTA = 2.0
HASH_KEY = 15485863

print(f"Using device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print("Model loaded.")

In [ ]:
vocab_size = len(tokenizer)

watermark_processor = WatermarkLogitsProcessor(
    vocab_size=vocab_size,
    gamma=GAMMA,
    delta=DELTA,
    hash_key=HASH_KEY,
)

detector = WatermarkDetector(
    vocab_size=vocab_size,
    gamma=GAMMA,
    hash_key=HASH_KEY,
    z_threshold=4.0,
)

## Quick demo: single prompt

In [ ]:
PROMPT = (
    "The quick brown fox jumps over the lazy dog. "
    "In recent news, scientists have discovered"
)

@torch.inference_mode()
def generate(prompt: str, watermarked: bool, max_new_tokens: int = 200) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    processors = [watermark_processor] if watermarked else []
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        logits_processor=processors,
    )
    new_tokens = out[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), new_tokens.tolist()

nw_text, nw_tokens = generate(PROMPT, watermarked=False)
w_text,  w_tokens  = generate(PROMPT, watermarked=True)

print("=== No watermark ===")
print(nw_text[:300])
print()
print("=== Watermarked ===")
print(w_text[:300])

In [ ]:
nw_result = detector.detect(nw_tokens)
w_result  = detector.detect(w_tokens)

print("No-watermark detection:", nw_result)
print("Watermarked detection: ", w_result)

## Bulk generation on C4

Run from terminal (slow — downloads model + dataset):
```bash
python generate.py --num_samples 500 --output_file data/generations.jsonl
```

Then evaluate:
```bash
python evaluate.py --input_file data/generations.jsonl
# add --compute_ppl to also measure oracle perplexity (requires OPT-2.7B)
```

## Entropy-aware extension (TODO)

Per the project proposal, the next step is a subclass of `WatermarkLogitsProcessor`
where `delta` is a function of the current token entropy rather than a fixed constant.